# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [9]:
"I am bucketing pages into a handful of recognizable situations, using signals from earlier weeks. Those are VISIBLE_BUT_AT_RISK, STRIKING_DISTANCE_DECLINING, LOW_VOLUME_UNPROVEN, and STABLE_HEALTHY_WATCH_LIST. Each archetype maps to one concrete next step which are refresh the content, review the snippet/title, just monitor it, or do nothing. Beacuse a short label so anyone reading the output later we, a teammate, our capstone reviewer can see why a page got flagged without rerunning the model."

print()
%pip -q install duckdb huggingface_hub
import duckdb
import pandas as pd
from google.colab import userdata
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

DEV_MONTH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
DIM_CLIENTS = "hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet"

labels = con.sql(f"""
    SELECT content_hash_id,
      AVG(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_clicks END) AS clicks_first_half,
      AVG(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_clicks END) AS clicks_second_half
    FROM read_parquet('{DEV_MONTH_PATH}')
    GROUP BY content_hash_id
""").df()
labels['is_declining'] = (labels['clicks_second_half'] < labels['clicks_first_half']).astype(int)

model_df = con.sql(f"""
    SELECT
      f.content_hash_id, f.client_hash_id,
      AVG(f.gsc_avg_position) AS avg_position,
      SUM(f.gsc_clicks) AS clicks_total,
      SUM(f.gsc_impressions) AS impressions_total,
      MAX(f.report_date) AS last_report_date
    FROM read_parquet('{DEV_MONTH_PATH}') f
    GROUP BY f.content_hash_id, f.client_hash_id
""").df()

dim = con.sql(f"""
    SELECT content_hash_id, content_type, word_count, main_intent,
           content_created_date
    FROM read_parquet('{DIM_CONTENT}')
""").df()

model_df = model_df.merge(dim, on='content_hash_id').merge(
    labels[['content_hash_id','is_declining']], on='content_hash_id'
)
model_df['content_age_days'] = (
    pd.to_datetime(model_df['last_report_date']) - pd.to_datetime(model_df['content_created_date'])
).dt.days
model_df = model_df.dropna(subset=['avg_position','word_count','content_age_days','client_hash_id'])

num_features = ['avg_position','word_count','content_age_days','impressions_total']
cat_features = ['content_type','main_intent']

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preproc = ColumnTransformer([
    ('num', 'passthrough', num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])

# Run with 3 different seeds to see if the gap direction is stable or just noise
for seed in [0, 1, 2]:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
    tr_idx, te_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
    tr, te = model_df.iloc[tr_idx], model_df.iloc[te_idx]
    m = Pipeline([('prep', preproc), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0))])
    m.fit(tr[num_features+cat_features], tr['is_declining'])
    auc = roc_auc_score(te['is_declining'], m.predict_proba(te[num_features+cat_features])[:,1])
    print(f"seed={seed}: grouped AUC = {auc:.3f}")

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# BEFORE: naive random row split (ignores that pages share clients)
X = model_df[num_features + cat_features]
y = model_df['is_declining']

X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(X, y, test_size=0.3, random_state=0)
rf_naive = Pipeline([('prep', preproc), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0))])
rf_naive.fit(X_tr_naive, y_tr_naive)
naive_auc = roc_auc_score(y_te_naive, rf_naive.predict_proba(X_te_naive)[:,1])

# AFTER: grouped split by client (what I used from Week 5 onward)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=0)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train_df, test_df = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_tr_grp, y_tr_grp = train_df[num_features + cat_features], train_df['is_declining']
X_te_grp, y_te_grp = test_df[num_features + cat_features], test_df['is_declining']

rf_grouped = Pipeline([('prep', preproc), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=0))])
rf_grouped.fit(X_tr_grp, y_tr_grp)
grouped_auc = roc_auc_score(y_te_grp, rf_grouped.predict_proba(X_te_grp)[:,1])

print(f"Naive random split AUC:   {naive_auc:.3f}")
print(f"Grouped-by-client AUC:    {grouped_auc:.3f}")
print(f"Gap (optimism from leakage): {naive_auc - grouped_auc:.3f}")

# Reusing model_df, rf_grouped, test_df (honest grouped-split model)
test_df = test_df.copy()
test_df['rf_prob'] = rf_grouped.predict_proba(test_df[num_features + cat_features])[:,1]
test_df['rf_pred'] = rf_grouped.predict(test_df[num_features + cat_features])

print(test_df['rf_prob'].describe())
print(test_df['rf_prob'].quantile([0.5, 0.75, 0.9, 0.95]))

p90 = test_df['rf_prob'].quantile(0.90)  # 0.405
p75 = test_df['rf_prob'].quantile(0.75)  # 0.182
p50 = test_df['rf_prob'].quantile(0.50)  # 0.087

def assign_archetype(row):
    if row['impressions_total'] < 50:
        return 'LOW_VOLUME_UNPROVEN'
    if row['avg_position'] <= 10 and row['rf_prob'] >= p90:
        return 'VISIBLE_BUT_AT_RISK'
    elif 10 < row['avg_position'] <= 20 and row['rf_prob'] >= p75:
        return 'STRIKING_DISTANCE_DECLINING'
    elif row['rf_prob'] < p50:
        return 'STABLE_HEALTHY'
    else:
        return 'WATCH_LIST'

test_df['archetype'] = test_df.apply(assign_archetype, axis=1)
test_df['action'] = test_df['archetype'].map(lambda a: archetype_action_map[a][0])
test_df['reason_code'] = test_df['archetype']
print(test_df['archetype'].value_counts())

archetype_action_map = {
    'VISIBLE_BUT_AT_RISK':        ('CTR_AND_SNIPPET_REVIEW', 'Already ranks well (pos ≤10) but model flags decline risk — protect existing visibility first, per paper Finding #3.'),
    'STRIKING_DISTANCE_DECLINING':('CONTENT_REFRESH',        'Pos 11-20 with decline risk — refresh + internal links to push toward page 1, per paper Finding #3 recommendation.'),
    'LOW_VOLUME_UNPROVEN':        ('MONITOR_ONLY',           'Too little traffic (<50 impressions) to trust CTR/decline signal — avoid acting on noise (lesson from Week 4).'),
    'STABLE_HEALTHY':             ('NO_ACTION',              'Model rates low decline risk — leave alone, don\'t spend review budget here.'),
    'WATCH_LIST':                 ('QUARTERLY_REVIEW',       'Ambiguous signal — neither clearly healthy nor clearly at risk; revisit next cycle.'),
}

ranked = test_df.sort_values('rf_prob', ascending=False)[
    ['content_hash_id','client_hash_id','archetype','action','reason_code','rf_prob',
     'avg_position','impressions_total','word_count','content_age_days']
]
ranked.head(15)

#Over half the scored portfolio (12,878 of 24,684 pages) falls below the 50-impression floor and is excluded from action recommendations entirely. This isn't a flaw in the model — insufficient traffic genuinely can't support a reliable decline signal — but it does mean this playbook is only actionable for the more visible half of the portfolio. A separate, lower-stakes process (e.g. periodic spot-checks) may be needed for the low-volume tail.

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

seed=0: grouped AUC = 0.848
seed=1: grouped AUC = 0.785
seed=2: grouped AUC = 0.814
Naive random split AUC:   0.837
Grouped-by-client AUC:    0.848
Gap (optimism from leakage): -0.011
count    24684.000000
mean         0.149810
std          0.134670
min          0.005731
25%          0.066105
50%          0.091487
75%          0.181820
max          0.530892
Name: rf_prob, dtype: float64
0.50    0.091487
0.75    0.181820
0.90    0.403493
0.95    0.475994
Name: rf_prob, dtype: float64
archetype
LOW_VOLUME_UNPROVEN            12878
WATCH_LIST                      7124
VISIBLE_BUT_AT_RISK             1776
STABLE_HEALTHY                  1664
STRIKING_DISTANCE_DECLINING     1242
Name: count, dtype: int64


,content_hash_id,client_hash_id,archetype,action,reason_code,rf_prob,avg_position,impressions_total,word_count,content_age_days
154740,content_72524cabb2854075,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.530892,4.540522,24413.0,2340,56
112427,content_eaccd7441ee37d02,client_a2eeb8899886adde,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.528635,5.623897,4034.0,1994,277
320248,content_36f7ea4dc82ac90f,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.528086,6.481745,5597.0,2288,78
51999,content_e7fd14f59717fbe0,client_a80fca3f171ed1de,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.528069,5.230385,5542.0,2679,109
154825,content_bc45d920e3b98949,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.527960,4.980995,9367.0,2375,54
155009,content_6cb14419bcc7e2d5,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.527802,4.489742,39567.0,2524,49
154994,content_6dda00a7fc03a82c,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.527607,4.101299,10378.0,2315,49
154861,content_f01b23a8993d265f,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.527461,4.421005,13410.0,2035,54
218627,content_2f37b6df11ab28bb,client_a80fca3f171ed1de,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.527230,4.683968,22560.0,2570,77
154637,content_d47190cdf7c509f7,client_e5c2aa26a8598242,VISIBLE_BUT_AT_RISK,CTR_AND_SNIPPET_REVIEW,VISIBLE_BUT_AT_RISK,0.527208,5.414299,20179.0,2515,70


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [3]:
"A prioritization aid for a content team deciding which pages to review this cycle. But it is trained on one month of one client cohort's _sample slice and not validated against the sealed June 2026 month or the full multi-year table. Furthermore, Proxy label (is_declining) is FlyRank's own derived comparison, not truth."

"A prioritization aid for a content team deciding which pages to review this cycle. But it is trained on one month of one client cohort's _sample slice and not validated against the sealed June 2026 month or the full multi-year table. Furthermore, Proxy label (is_declining) is FlyRank's own derived comparison, not truth."

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [4]:
"A person must check that any page where impressions_total is near the 50-impression floor, check it isn't a seasonal or recently-published page miscategorized as low volume. And there must be no auto publishing or auto editing of content based on the model's output."

"A person must check that any page where impressions_total is near the 50-impression floor, check it isn't a seasonal or recently-published page miscategorized as low volume. And there must be no auto publishing or auto editing of content based on the model's output."

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [5]:
"if LOW_VOLUME_UNPROVEN starts growing as a share of the queue, it may mean impression thresholds need recalibrating. Moreover, re run monthly as new month=YYYY-MM partitions land, since a single month may not represent seasonal variation."

'if LOW_VOLUME_UNPROVEN starts growing as a share of the queue, it may mean impression thresholds need recalibrating. Moreover, re run monthly as new month=YYYY-MM partitions land, since a single month may not represent seasonal variation.'

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
import os, json
os.makedirs('work/outputs', exist_ok=True)

ranked.to_csv('work/outputs/action_playbook_queue.csv', index=False)

metrics = {
    'grouped_split_auc': float(grouped_auc),
    'naive_split_auc': float(naive_auc),
    'leakage_gap': float(naive_auc - grouped_auc),
    'archetype_counts': test_df['archetype'].value_counts().to_dict(),
    'n_scored_pages': int(len(test_df)),
    'dev_month': '2026-03',
}
with open('work/outputs/w07_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("Exported queue:", len(ranked), "rows")
print(json.dumps(metrics, indent=2))

Exported queue: 24684 rows
{
  "grouped_split_auc": 0.8483241578595602,
  "naive_split_auc": 0.8372363702981899,
  "leakage_gap": -0.0110877875613703,
  "archetype_counts": {
    "LOW_VOLUME_UNPROVEN": 12878,
    "WATCH_LIST": 7124,
    "VISIBLE_BUT_AT_RISK": 1776,
    "STABLE_HEALTHY": 1664,
    "STRIKING_DISTANCE_DECLINING": 1242
  },
  "n_scored_pages": 24684,
  "dev_month": "2026-03"
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.